# Chelsea's Vesicle Analysis Script

This script detects vesicles (GUVs) in microscope images and measures the fluorescence of every vesicle.  
It can measure either whole vesicle fluorescence or just membrane localised fluorescence.  
The main output is a violin plot of the measured fluorescence of all vesicles in each condition, pooled and for each repeat.  
Various checkpoints are integrated which can be used to troubleshoot and check accuracy.

Data must be in this format:
- One folder per condition (eg. 'noNTPs_t0', 'withNTPs_t2'), each containing all of the repeats for that condition.
- Colour or greyscale images must be saved as '.png'.
- Each unique image must have these images with these naming patterns ('repeat. condition'):
    Eg. 'A. noNTPs_t0_DYE.png'        = Image of vesicle marker dye, used to detect the vesicles (repeat A).
        'A. noNTPs_t0_MES.png'        = Image of fluorescence to measure.
        'A. noNTPs_t0_Background.png' = Image of representative negative background for normalisation
                                        (not needed for 'MEMBRANE-INNER').
- The folder names will be used as graph labels as default, but these can be edited.

To use this notebook:
1. Install the packages once with: `pip install numpy scipy scikit-image matplotlib pandas pillow seaborn ipykernel ipympl`  
   To use Cellpose detection, also install it once with: `pip install cellpose`
2. Open in VS Code, choose a Python kernel (top right) and run each cell in order with Shift+Enter.
3. Set working_folder below to the folder containing the condition folders.
4. Follow the instructions in the code below.

Figures are interactive: use the buttons on each figure to zoom & pan (the home button resets the view), like MATLAB figures.

## INPUT PARAMETERS

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

import matplotlib.pyplot as plt

# interactive figures (zoom & pan), change 'widget' to 'inline' for plain images
%matplotlib widget
plt.close('all')  # close current figures

# define working directory (the folder containing the condition folders) CHANGE THIS
# either a full path eg. '/Users/name/Documents/Data_vesicle', or a path relative to this notebook's folder ('..' = the folder above)
working_folder = '../Data_vesicle'

# the folder containing this notebook, remembered so this cell can be re-run after it changes to the working folder below
if 'notebook_folder' not in globals():
    notebook_folder = os.getcwd()
scripts_folder = os.path.join(notebook_folder, 'Functions')  # the function scripts, found automatically
working_folder = os.path.abspath(os.path.join(notebook_folder, working_folder))

for folder in [scripts_folder, working_folder]:
    if not os.path.isdir(folder):
        raise FileNotFoundError('Folder not found: %s\nCheck working_folder is correct, and the Functions folder is next to this notebook.' % folder)
sys.path.insert(0, scripts_folder)
os.chdir(working_folder)

# load required functions
from func_detect_measure_vesicle import func_detect_measure_vesicle, measure_labels
from func_line_profile_vesicle import func_line_profile_vesicle
from func_plot_MES_for_sample_vesicle import func_plot_MES_for_sample_vesicle
from func_summary_tables_vesicle import func_summary_tables_vesicle
from func_test_detection_vesicle import func_test_detection_vesicle
from func_violin_plot import func_report_off_axis, func_style_axes, func_violin_plot

# define the condition folders to process, in the order they will be plotted
condition_folders = ['noNTPs_t0', 'noNTPs_t2', 'withNTPs_t0', 'withNTPs_t2']

# Input the fluorescent region you want to measure (the violin plot axis label changes to match):
#   - 'WHOLE' = averages fluorescence of the whole vesicle, then subtracts background.
#   - 'MEMBRANE-BACKGROUND' = averages fluorescence of the membrane using line profiles, then subtracts background.
#   - 'MEMBRANE-INNER' = averages fluorescence of the membrane using line profiles, then divides by the average inside the vesicle (ratio membrane/lumen).
measure_fluorescence_type = 'MEMBRANE-INNER'

# Input the method used to detect the vesicles in the marker dye image:
#   - 'HOUGH' = circular Hough transform, the same as MATLAB's imfindcircles (the original method).
#   - 'CELLPOSE' = Cellpose deep learning segmentation. Slower (about 1 minute per image the first time, then saved).
detection_method = 'HOUGH'

# Input the parameters used to detect the vesicles:
pixel_size_micron = 124.5 / 1920  # size of one pixel in microns (image width in microns / width in pixels), set for your microscope
sensitivity_factor = 0.92  # Only used by 'HOUGH'. Sensitivity ranges from 0 (Perfect circle) to 1 (Barely a circle)
min_diameter_micron = 5  # Minimum vesicle diameter in microns, Nb. below 5 min diameter can skew results.
max_diameter_micron = 22  # Maximum vesicle diameter in microns

# If measuring membrane fluorescence:
# The script will take 4 line profiles (cross-sections) per vesicle, from the centre outwards.
# It will find the peak fluorescence of each within a defined region where the membrane should be found, then average the two middle values.
# Define where the membrane should be found along the vesicle's radius:
inner_boundary = 0.85  # position of the inner membrane boundary in multiples of the radius
outer_boundary = 1.1  # position of the outer membrane boundary in multiples of the radius, max is 1.3

# If measuring 'MEMBRANE-INNER':
# Vesicles with a ratio above this are removed from all results (tables & plots).
# A near-black inner circle gives a meaningless, extreme ratio (eg. dividing by 1 gives 100s-1000s),
# while real, strongly localised vesicles in this dataset reach ratios of about 26.
max_ratio = 30

# collects the settings above to pass to the functions (no need to edit)
settings = dict(measure_fluorescence_type=measure_fluorescence_type, detection_method=detection_method,
                pixel_size_micron=pixel_size_micron, sensitivity_factor=sensitivity_factor,
                min_diameter_micron=min_diameter_micron, max_diameter_micron=max_diameter_micron,
                inner_boundary=inner_boundary, outer_boundary=outer_boundary, max_ratio=max_ratio)

# NOW RUN THIS SECTION ^

## TEST PARAMETERS

In [ ]:
# Check the vesicle detection / membrane parameters are suitable for your samples and images.
# For a specified test sample, a figure will show where it detected vesicles and where it will measure fluorescence.

# Specify the condition folder and the individual image you want to test (file name without '_DYE.png'):
test_condition = 'noNTPs_t0'
test_sample = 'A. noNTPs_t0'

# Create the figure
func_test_detection_vesicle(test_condition, test_sample, settings)
plt.show()

# NOW RUN THIS SECTION ^
# Adjust parameters defined above accordingly, then re-run INPUT PARAMETERS and this section.

## DETECT & MEASURE FLUORESCENCE FOR ALL SAMPLES

In [ ]:
# For all image files in every condition folder, detect vesicles and quantify fluorescence.
# All repeats within a condition folder are pooled together.
# Re-run this section whenever you change the INPUT PARAMETERS. The check sections below will give an error if you forget.
vesicleData, imageList = func_detect_measure_vesicle(condition_folders, settings)

# NOW RUN THIS SECTION ^
# The measurements for every vesicle are stored in the 'vesicleData' table (one row per vesicle):
#       Condition, Image, Repeat, Vesicle (number shown on the check plots), X, Y (pixels), Radius (pixels), MES (measured fluorescence)

# vesicleData.to_csv('all_vesicles.csv', index=False)  # UNCOMMENT to save every vesicle as a spreadsheet

## NUMBER, MEAN FLUORESCENCE & % LOCALISED GUVS FOR EACH CONDITION & IMAGE

In [ ]:
# Number of GUVs and mean & median measured fluorescence, for each condition (all repeats pooled) & each image (ie. each repeat)
# For 'MEMBRANE-INNER', also the number & % of GUVs localised (membrane/lumen ratio > 1)
# Includes every measured vesicle (the plot cut off in the section below does not apply here)
conditionTable, imageTable = func_summary_tables_vesicle(vesicleData, imageList)
print('EACH CONDITION (all repeats pooled):')
print(conditionTable.to_string(), end='\n\n')
print('EACH IMAGE (ie. each repeat):')
print(imageTable.to_string())

# conditionTable.to_csv('summary_per_condition.csv')  # UNCOMMENT to save as a spreadsheet
# imageTable.to_csv('summary_per_image.csv', index=False)  # UNCOMMENT to save as a spreadsheet

# NOW RUN THIS SECTION ^

## MAKE VIOLIN PLOTS

In [ ]:
# plot measured fluorescence values for each condition onto a violin plot, all repeats pooled
#       - boxplots indicate the interquartile range
#       - notches indicate the median
#       - asterisks indicate the mean
#       - the violins use every measured vesicle, the same data as the tables
# edit y axis limits, font size, titles, colours, etc below as desired

# colours for the first 4 conditions, more are added automatically if there are more conditions (or add your own here)
violinColors = [
    (0.0, 0.4470, 0.7410),     # blue
    (0.8660, 0.3290, 0),       # (0.8190, 0.0150, 0.5450) pink
    (0.1840, 0.7450, 0.9370),  # light blue
    (0.2310, 0.6660, 0.1960),  # green
]

yLabel = measure_labels[measure_fluorescence_type]  # y axis label, set automatically from the measurement type (or type your own label here)
# y axis limits: lower yMax to stop a few sparse high values stretching the plot.
# Points outside the limits are hidden from view only, they are still included in the violin, box & mean. It prints how many are hidden.
yMax = 4.5  # define y axis maximum CHANGE THIS, or None for automatic
yMin = 0  # define y axis minimum CHANGE THIS, or None for automatic
func_report_off_axis(vesicleData, 'MES', yMin, yMax)

# Create violin plot grouped by condition
figure2, ax1 = plt.subplots(figsize=(5, 7))
func_violin_plot(vesicleData, 'MES', condition_folders, violinColors, ax1, point_alpha=1)  # point_alpha=0 hides individual data points

ax1.set_ylabel(yLabel)
ax1.set_ylim([yMin, yMax])
#ax1.set_xticklabels(['- SLO, T=0', '- SLO, T=2', '+ SLO, T=0', '+ SLO, T=2'])  # UNCOMMENT to manually define x axis labels
func_style_axes(ax1)
plt.tight_layout()
plt.show()

# NOW RUN THIS SECTION ^
# Plots can be saved with figure2.savefig('violin.png', dpi=300)

In [ ]:
# Same violin plot for each repeat, side by side for easy comparison (uses the colours & y axis limits from the section above).
# The repeat is the start of each image name, before the first '.' (eg. 'A' in 'A. noNTPs_t0').
repeats = sorted(vesicleData['Repeat'].unique())

figureR, axes = plt.subplots(1, len(repeats), figsize=(3.6 * len(repeats), 7), sharey=True, squeeze=False)
for ax1, repeat in zip(axes[0], repeats):
    func_violin_plot(vesicleData[vesicleData['Repeat'] == repeat], 'MES', condition_folders, violinColors, ax1, point_alpha=1)
    ax1.set_ylim([yMin, yMax])
    ax1.set_title('Repeat ' + repeat, fontsize=18)
    ax1.tick_params(labelleft=True)  # show the y axis numbers on every plot
    func_style_axes(ax1)
axes[0][0].set_ylabel(yLabel)
plt.tight_layout()
plt.show()

# NOW RUN THIS SECTION ^
# Plots can be saved with figureR.savefig('violin_repeats.png', dpi=300)
# Use the below sections to check assigned values for individual samples & vesicles
# Adjust detection parameters, images, etc & repeat process, if desired

## CHECK MEASURED FLUORESCENCE IS AS EXPECTED

In [ ]:
# This will overlay a coloured spot onto each vesicle in the specified image, indicating its measured fluorescence.
# Each vesicle is labelled with its number, for use in the line profile section below.
# Used to check the vesicles are being quantified as expected.
# Uses the vesicles from the DETECT & MEASURE section, so run that first.

# Specify the condition folder and the individual image you want to check (file name without '_DYE.png'):
test_condition = 'withNTPs_t2'
test_sample = 'A. withNTPs_t2'

# range of measured fluorescence values for the colour scale:
maxyvalue = yMax  # define maximum value, default is same as violin plot axis (None = automatic)
minyvalue = yMin  # define minimum value, default is same as violin plot axis (None = automatic)

# plot the image
func_plot_MES_for_sample_vesicle(test_condition, test_sample, vesicleData, settings, maxyvalue, minyvalue)
plt.show()

## CHECK MEMBRANE LINE PROFILE VALUES FOR INDIVIDUAL VESICLES

In [ ]:
# This will plot the raw membrane line profile data for a specified vesicle in a specified sample (right),
# next to the image with the vesicle's 4 line profiles drawn in the same colours (left, zoom in to see it).
#      - Red dashed vertical lines indicate the membrane region searched for peak fluorescence.
#      - Red horizontal line is the average (of the two middle values) peak membrane fluorescence.
#      - Green dashed vertical line indicates the inner vesicle region used.
#      - Green horizontal line is the average inner vesicle fluorescence value.
# Uses the vesicles from the DETECT & MEASURE section, so run that first.

# Specify the condition folder and the individual image you want to check:
test_condition = 'noNTPs_t2'
test_sample = 'A. noNTPs_t2'
# Specify the number of the vesicle you want to check (shown on each vesicle by the section above):
k = 7

# create the plot
func_line_profile_vesicle(test_condition, test_sample, k, vesicleData, settings)
plt.show()